In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║          TCGA-BRCA SURVIVAL ANALYSIS & PREDICTION PIPELINE                 ║
║          Dataset : data_clinical_patient.txt (cBioPortal)                  ║
║          Patients: 1,084  |  Features: 38  |  Target: Overall Survival     ║
║                                                                             ║
║  Pipeline Stages:                                                           ║
║    1.  Data Loading & Validation                                            ║
║    2.  Preprocessing & Imputation                                           ║
║    3.  Feature Engineering                                                  ║
║    4.  Exploratory Data Analysis (EDA)                                      ║
║    5.  Kaplan-Meier Analysis (OS, Subtype, Stage, Ancestry, Radiation)     ║
║    6.  Cox Proportional Hazards Model + Forest Plot                        ║
║    7.  5-Year Survival Classification Label                                 ║
║    8.  Model Training & 5-Fold CV (LR / RF / GB / MLP / XGB)              ║
║    9.  Evaluation Plots (ROC, CM, PR, Comparison)                          ║
║   10.  Risk Stratification (KM + score distribution)                       ║
║   11.  SHAP / Permutation Explainability                                   ║
║   12.  Calibration Curves                                                  ║
║   13.  Summary Report                                                      ║
╚══════════════════════════════════════════════════════════════════════════════╝

  DATASET : TCGA-BRCA clinical data (cBioPortal)
  PYTHON  : 3.9+
  KEY LIBS: lifelines, sklearn, xgboost, shap, matplotlib, seaborn
"""

# ══════════════════════════════════════════════════════════════════════════════
# 0 · IMPORTS & SETUP
# ══════════════════════════════════════════════════════════════════════════════
import os, sys, warnings, logging, textwrap, time
from pathlib import Path
!pip install lifelines
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

# Survival analysis
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

# Machine learning
from sklearn.model_selection  import (StratifiedKFold, cross_validate,
                                       cross_val_predict)
from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import (RandomForestClassifier,
                                       GradientBoostingClassifier)
from sklearn.neural_network   import MLPClassifier
from sklearn.preprocessing    import StandardScaler, label_binarize
from sklearn.pipeline         import Pipeline
from sklearn.calibration      import CalibratedClassifierCV, calibration_curve
from sklearn.metrics          import (roc_auc_score, average_precision_score,
                                       f1_score, accuracy_score,
                                       confusion_matrix, ConfusionMatrixDisplay,
                                       roc_curve, precision_recall_curve,
                                       brier_score_loss)
from sklearn.inspection       import permutation_importance

# SHAP (optional)
try:
    import shap
    SHAP_OK = True
except ImportError:
    SHAP_OK = False
    print("⚠  pip install shap  for SHAP explainability plots")

# XGBoost (optional)
try:
    from xgboost import XGBClassifier
    XGB_OK = True
except ImportError:
    XGB_OK = False
    print("⚠  pip install xgboost  for XGBoost model")

warnings.filterwarnings("ignore")
logging.getLogger("lifelines").setLevel(logging.ERROR)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Output directories ────────────────────────────────────────────────────────
ROOT  = Path("tcga_brca_results")
PLOTS = ROOT / "plots"
DATA  = ROOT / "data"
for d in [PLOTS, DATA]:
    d.mkdir(parents=True, exist_ok=True)

# ── Publication-grade plot theme ──────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi"         : 150,  "savefig.dpi"        : 300,
    "figure.facecolor"   : "white", "axes.facecolor"   : "#F8F9FA",
    "axes.spines.top"    : False,   "axes.spines.right" : False,
    "axes.spines.left"   : True,    "axes.spines.bottom": True,
    "axes.edgecolor"     : "#CCCCCC","axes.linewidth"   : 0.8,
    "axes.titlesize"     : 13,   "axes.labelsize"     : 11,
    "axes.titlepad"      : 10,
    "xtick.labelsize"    : 9,    "ytick.labelsize"    : 9,
    "xtick.direction"    : "out","ytick.direction"    : "out",
    "legend.fontsize"    : 9,    "legend.framealpha"  : 0.88,
    "legend.edgecolor"   : "#CCCCCC",
    "lines.linewidth"    : 2.2,  "font.family"        : "DejaVu Sans",
    "grid.color"         : "#E5E5E5","grid.linewidth"  : 0.6,
    "axes.grid"          : True, "grid.alpha"         : 0.6,
})
PAL  = sns.color_palette("tab10")
PAL6 = sns.color_palette("Set2", 6)
CMAP = "RdYlBu_r"

# ── Column mapping ────────────────────────────────────────────────────────────
OS_TIME   = "OS_MONTHS"
OS_EVENT  = "OS_STATUS"      # "0:LIVING" / "1:DECEASED"
DFS_TIME  = "DFS_MONTHS"
DFS_EVENT = "DFS_STATUS"     # "0:DiseaseFree" / "1:Recurred/Progressed"

STAGE_MAP = {
    "STAGE I"  :"Stage_I",  "STAGE IA" :"Stage_I",  "STAGE IB" :"Stage_I",
    "STAGE II" :"Stage_II", "STAGE IIA":"Stage_II", "STAGE IIB":"Stage_II",
    "STAGE III":"Stage_III","STAGE IIIA":"Stage_III","STAGE IIIB":"Stage_III",
    "STAGE IIIC":"Stage_III","STAGE IV" :"Stage_IV",
    "STAGE X"  :"Unknown",  "[NOT AVAILABLE]":"Unknown","NAN":"Unknown",
}

CLINICAL_FEATURES = [
    "AGE","SUBTYPE","AJCC_PATHOLOGIC_TUMOR_STAGE",
    "PATH_T_STAGE","PATH_N_STAGE","PATH_M_STAGE",
    "RADIATION_THERAPY","RACE","ETHNICITY","GENETIC_ANCESTRY_LABEL",
    "PERSON_NEOPLASM_CANCER_STATUS","HISTORY_NEOADJUVANT_TRTYN",
    "NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT","PRIOR_DX",
    "PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT",
]

NA_STRINGS = {"nan","NaN","NA","N/A","","[Not Available]",
              "[Not Applicable]","[Unknown]","[Discrepancy]"}


# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _section(title: str):
    print(f"\n{'─'*68}")
    print(f"  {title}")
    print(f"{'─'*68}")


def _save(fig, name: str, section=""):
    path = PLOTS / name
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    label = f"{section}  " if section else ""
    print(f"  ✔ Saved {label}{name}")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 · LOAD & VALIDATE
# ══════════════════════════════════════════════════════════════════════════════

def load_and_validate(filepath: str) -> pd.DataFrame:
    _section("SECTION 1 · Loading & Validating Data")

    df = pd.read_csv(filepath, sep="\t", comment="#", low_memory=False)
    df.columns = df.columns.str.strip()

    print(f"  ✔ Loaded       : {df.shape[0]:,} patients × {df.shape[1]} columns")
    print(f"  ✔ File size    : {os.path.getsize(filepath):,} bytes")

    mandatory = [OS_TIME, OS_EVENT, DFS_TIME, DFS_EVENT, "SUBTYPE",
                 "AJCC_PATHOLOGIC_TUMOR_STAGE", "GENETIC_ANCESTRY_LABEL"]
    missing   = [c for c in mandatory if c not in df.columns]
    if missing:
        raise ValueError(f"Missing mandatory columns: {missing}")
    print(f"  ✔ All mandatory columns present")

    print(f"\n  OS_STATUS distribution:")
    for k, v in df[OS_EVENT].value_counts().items():
        print(f"    {str(k)!r:35s} → {v}")

    return df


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 · PREPROCESSING
# ══════════════════════════════════════════════════════════════════════════════

def preprocess(df: pd.DataFrame):
    _section("SECTION 2 · Preprocessing")
    df = df.copy()

    df["OS_EVENT_BIN"] = (
        df[OS_EVENT].astype(str).str.strip().str.upper()
        .str.startswith("1").astype(int)
    )
    df["DFS_EVENT_BIN"] = (
        df[DFS_EVENT].astype(str).str.strip()
        .apply(lambda x: 1 if x.startswith("1") else 0)
    )

    df["OS_MONTHS"]       = pd.to_numeric(df[OS_TIME],  errors="coerce")
    df["DFS_MONTHS_CLEAN"] = pd.to_numeric(df[DFS_TIME], errors="coerce")

    n0 = len(df)
    df = df.dropna(subset=["OS_MONTHS", "OS_EVENT_BIN"])
    df = df[df["OS_MONTHS"] > 0]
    print(f"  ✔ Dropped {n0 - len(df)} rows with missing/zero OS")

    ec = df["OS_EVENT_BIN"].value_counts().sort_index()
    print(f"  ✔ OS events    : LIVING={ec.get(0,0)}  DECEASED={ec.get(1,0)}")
    print(f"  ✔ Event rate   : {ec.get(1,0)/len(df)*100:.1f}%")
    print(f"  ✔ Median OS    : {df['OS_MONTHS'].median():.1f} months")
    print(f"  ✔ Median age   : {pd.to_numeric(df['AGE'], errors='coerce').median():.0f} years")

    keep = [c for c in CLINICAL_FEATURES if c in df.columns]
    df   = df[["OS_MONTHS","OS_EVENT_BIN","DFS_MONTHS_CLEAN","DFS_EVENT_BIN"]
               + keep].copy()

    df["AJCC_PATHOLOGIC_TUMOR_STAGE"] = (
        df["AJCC_PATHOLOGIC_TUMOR_STAGE"].astype(str).str.upper().str.strip()
        .map(lambda x: STAGE_MAP.get(x, "Unknown"))
    )

    cat_cols = df.select_dtypes("object").columns
    for c in cat_cols:
        df[c] = df[c].astype(str).str.strip().replace(NA_STRINGS, "Unknown")

    df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
    df["AGE"] = df["AGE"].fillna(df["AGE"].median())

    print(f"  ✔ Final shape  : {df.shape}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 · FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════════════════

def engineer_features(df: pd.DataFrame):
    _section("SECTION 3 · Feature Engineering")
    df = df.copy()

    surv_cols = ["OS_MONTHS","OS_EVENT_BIN","DFS_MONTHS_CLEAN","DFS_EVENT_BIN"]
    cat_cols  = [c for c in df.select_dtypes("object").columns if c not in surv_cols]

    df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)
    feat   = [c for c in df_enc.columns if c not in surv_cols]

    # Remove constants
    var   = df_enc[feat].var()
    const = var[var == 0].index.tolist()
    df_enc.drop(columns=const, inplace=True)
    feat  = [c for c in feat if c not in const]

    # Remove high collinearity (|r| > 0.95)
    corr  = df_enc[feat].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    hc    = [c for c in upper.columns if any(upper[c] > 0.95)]
    df_enc.drop(columns=hc, inplace=True)
    feat  = [c for c in feat if c not in hc]

    print(f"  ✔ Removed {len(const)} constant + {len(hc)} collinear columns")
    print(f"  ✔ Final feature count : {len(feat)}")
    print(f"  ✔ Feature sample      : {feat[:8]} …")
    return df_enc, feat


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 · EDA PLOTS
# ══════════════════════════════════════════════════════════════════════════════

def plot_eda(df: pd.DataFrame, raw: pd.DataFrame):
    _section("SECTION 4 · Exploratory Data Analysis")

    fig = plt.figure(figsize=(20, 12))
    gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.50, wspace=0.42)

    # 1. Age histogram
    ax = fig.add_subplot(gs[0, 0])
    ages = pd.to_numeric(raw["AGE"], errors="coerce").dropna()
    ax.hist(ages, bins=25, color=PAL[0], edgecolor="white", alpha=0.88)
    ax.axvline(ages.median(), color="crimson", lw=2, ls="--",
               label=f"Median {ages.median():.0f} yr")
    ax.set_title("Age at Diagnosis", fontweight="bold")
    ax.set_xlabel("Age (years)"); ax.legend()

    # 2. OS event donut
    ax = fig.add_subplot(gs[0, 1])
    ec = df["OS_EVENT_BIN"].value_counts().sort_index()
    wedges, texts, autotexts = ax.pie(
        ec.values, labels=["Living", "Deceased"],
        colors=[PAL[2], PAL[3]], autopct="%1.1f%%",
        startangle=90, wedgeprops=dict(edgecolor="white", width=0.55))
    [t.set_fontsize(10) for t in autotexts]
    ax.set_title("OS Event Distribution", fontweight="bold")
    ax.grid(False)

    # 3. OS months distribution
    ax = fig.add_subplot(gs[0, 2])
    ax.hist(df["OS_MONTHS"], bins=30, color=PAL[4], edgecolor="white", alpha=0.88)
    ax.axvline(60, color="crimson", lw=2, ls="--", label="60-month cut")
    ax.set_title("Follow-up (OS Months)", fontweight="bold")
    ax.set_xlabel("Months"); ax.legend()

    # 4. Subtype bar
    ax = fig.add_subplot(gs[0, 3])
    if "SUBTYPE" in raw.columns:
        sc = raw["SUBTYPE"].value_counts()
        bars = ax.barh(sc.index[::-1], sc.values[::-1],
                       color=PAL6[:len(sc)], edgecolor="white")
        for bar, v in zip(bars, sc.values[::-1]):
            ax.text(v + 5, bar.get_y() + bar.get_height() / 2,
                    str(v), va="center", fontsize=8)
        ax.set_title("Molecular Subtype", fontweight="bold")

    # 5. AJCC stage
    ax = fig.add_subplot(gs[1, 0])
    stg_order = ["Stage_I","Stage_II","Stage_III","Stage_IV","Unknown"]
    stg = df["AJCC_PATHOLOGIC_TUMOR_STAGE"].value_counts().reindex(
        [s for s in stg_order if s in df["AJCC_PATHOLOGIC_TUMOR_STAGE"].values])
    stg.plot(kind="bar", ax=ax, color=PAL[:len(stg)], edgecolor="white", alpha=0.88)
    ax.set_title("AJCC Stage Distribution", fontweight="bold")
    ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
    for p in ax.patches:
        ax.annotate(str(int(p.get_height())),
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha="center", va="bottom", fontsize=8)

    # 6. Genetic ancestry
    ax = fig.add_subplot(gs[1, 1])
    if "GENETIC_ANCESTRY_LABEL" in raw.columns:
        ga = raw["GENETIC_ANCESTRY_LABEL"].value_counts()
        ax.pie(ga.values, labels=ga.index,
               colors=PAL6[:len(ga)], autopct="%1.1f%%",
               startangle=90, wedgeprops=dict(edgecolor="white"))
        ax.set_title("Genetic Ancestry", fontweight="bold")
        ax.grid(False)

    # 7. Radiation therapy
    ax = fig.add_subplot(gs[1, 2])
    if "RADIATION_THERAPY" in raw.columns:
        rt = raw["RADIATION_THERAPY"].value_counts()
        colors_rt = [PAL[1], PAL[3], PAL[7]][:len(rt)]
        ax.bar(rt.index, rt.values, color=colors_rt, edgecolor="white")
        ax.set_title("Radiation Therapy", fontweight="bold")
        for p in ax.patches:
            ax.annotate(str(int(p.get_height())),
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha="center", va="bottom", fontsize=9)

    # 8. Tumor status
    ax = fig.add_subplot(gs[1, 3])
    if "PERSON_NEOPLASM_CANCER_STATUS" in raw.columns:
        ts = raw["PERSON_NEOPLASM_CANCER_STATUS"].value_counts()
        ax.bar(ts.index, ts.values,
               color=[PAL[2], PAL[3]][:len(ts)], edgecolor="white")
        ax.set_title("Tumor Status at Last Follow-up", fontweight="bold")
        ax.tick_params(axis="x", rotation=15)
        for p in ax.patches:
            ax.annotate(str(int(p.get_height())),
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha="center", va="bottom", fontsize=9)

    fig.suptitle("TCGA-BRCA — Exploratory Data Analysis  (n=1,084)",
                 fontsize=16, fontweight="bold", y=1.01)
    _save(fig, "00_eda_overview.png")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 · KAPLAN-MEIER
# ══════════════════════════════════════════════════════════════════════════════

def _km_ax(ax, T, E, groups, palette, title, pval=True, ci=True):
    """Reusable KM plotter. Returns log-rank p-value."""
    Ts, Es, labs = [], [], []
    for i, (label, mask) in enumerate(groups.items()):
        if mask.sum() < 10:
            continue
        kmf = KaplanMeierFitter()
        kmf.fit(T[mask], E[mask], label=f"{label} (n={mask.sum()})")
        kmf.plot_survival_function(ax=ax, ci_show=ci,
                                   color=palette[i % len(palette)])
        Ts.append(T[mask]); Es.append(E[mask]); labs.append(label)

    pvalue = None
    if pval and len(Ts) >= 2:
        if len(Ts) == 2:
            pvalue = logrank_test(Ts[0], Ts[1], Es[0], Es[1]).p_value
        else:
            ct = pd.concat(Ts); ce = pd.concat(Es)
            cg = pd.concat([pd.Series([l] * len(t)) for l, t in zip(labs, Ts)])
            pvalue = multivariate_logrank_test(ct, cg, ce).p_value
        sig = "***" if pvalue < 0.001 else ("**" if pvalue < 0.01
              else ("*" if pvalue < 0.05 else "ns"))
        ax.set_title(f"{title}\nlog-rank p = {pvalue:.4f}  ({sig})",
                     fontweight="bold")
    else:
        ax.set_title(title, fontweight="bold")

    ax.set_xlabel("Time (Months)"); ax.set_ylabel("Survival Probability")
    ax.set_ylim(0, 1.05); ax.legend(loc="upper right", fontsize=8)
    return pvalue


def kaplan_meier_analysis(df: pd.DataFrame, raw: pd.DataFrame):
    _section("SECTION 5 · Kaplan-Meier Survival Analysis")

    T  = df["OS_MONTHS"]
    E  = df["OS_EVENT_BIN"]
    Td = df["DFS_MONTHS_CLEAN"].dropna()
    Ed = df["DFS_EVENT_BIN"].reindex(Td.index)

    # ── Plot A: Overall OS + Subtype ──────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    ax = axes[0]
    kmf = KaplanMeierFitter()
    kmf.fit(T, E, label=f"All patients (n={len(T)})")
    kmf.plot_survival_function(ax=ax, ci_show=True, color=PAL[0])
    med = kmf.median_survival_time_
    ax.axhline(0.5, color="grey", ls="--", lw=1, alpha=0.6)
    note = (f"Median OS: {med:.1f} mo" if not np.isinf(med)
            else "Median OS: Not reached")
    ax.text(0.97, 0.54, note, transform=ax.transAxes,
            ha="right", fontsize=9, color="grey")
    ax.set_title("Overall Survival — TCGA-BRCA (n=1,084)", fontweight="bold")
    ax.set_xlabel("Time (Months)"); ax.set_ylabel("Survival Probability")
    ax.set_ylim(0, 1.05)

    sub = raw["SUBTYPE"].reindex(df.index).astype(str).str.strip()
    valid_sub = [s for s in sub.value_counts().index
                 if sub.value_counts()[s] >= 20
                 and s not in ("Unknown","nan","")][:6]
    grp = {s: (sub == s) for s in valid_sub}
    pv  = _km_ax(axes[1], T, E, grp, PAL,
                 "OS by Molecular Subtype", pval=True, ci=False)
    print(f"  ✔ KM subtype log-rank p = {pv:.4f}" if pv else "  ✔ KM subtype (no p-value)")

    plt.suptitle("Kaplan-Meier — Overall Survival",
                 fontsize=15, fontweight="bold", y=1.01)
    plt.tight_layout()
    _save(fig, "01_km_overall_subtype.png")

    # ── Plot B: Stage + Ancestry ──────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    stage_order = ["Stage_I","Stage_II","Stage_III","Stage_IV"]
    stg_grp = {s: (df["AJCC_PATHOLOGIC_TUMOR_STAGE"] == s) for s in stage_order}
    pv2 = _km_ax(axes[0], T, E, stg_grp, PAL,
                 "OS by AJCC Tumour Stage", pval=True, ci=False)
    print(f"  ✔ KM stage log-rank p = {pv2:.4f}" if pv2 else "")

    anc = raw["GENETIC_ANCESTRY_LABEL"].reindex(df.index).astype(str).str.strip()
    valid_anc = [a for a in anc.value_counts().index
                 if anc.value_counts()[a] >= 15
                 and a not in ("Unknown","nan","")][:5]
    anc_grp = {a: (anc == a) for a in valid_anc}
    pv3 = _km_ax(axes[1], T, E, anc_grp, PAL6,
                 "OS by Genetic Ancestry", pval=True, ci=False)
    print(f"  ✔ KM ancestry log-rank p = {pv3:.4f}" if pv3 else "")

    plt.suptitle("Kaplan-Meier — Stage & Ancestry",
                 fontsize=15, fontweight="bold", y=1.01)
    plt.tight_layout()
    _save(fig, "02_km_stage_ancestry.png")

    # ── Plot C: DFS + Radiation ───────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    ax = axes[0]
    kmf2 = KaplanMeierFitter()
    kmf2.fit(Td, Ed, label=f"All patients (n={len(Td)})")
    kmf2.plot_survival_function(ax=ax, ci_show=True, color=PAL[5])
    ax.set_title("Disease-Free Survival — TCGA-BRCA", fontweight="bold")
    ax.set_xlabel("Time (Months)"); ax.set_ylabel("DFS Probability")
    ax.set_ylim(0, 1.05)

    rt_col = raw["RADIATION_THERAPY"].reindex(df.index).astype(str).str.strip()
    rt_grp = {r: (rt_col == r) for r in ["Yes","No"] if (rt_col == r).sum() >= 10}
    pv4 = _km_ax(axes[1], T, E, rt_grp, [PAL[1], PAL[3]],
                 "OS by Radiation Therapy", pval=True, ci=True)
    print(f"  ✔ KM radiation log-rank p = {pv4:.4f}" if pv4 else "")

    plt.suptitle("Disease-Free Survival & Radiation Effect",
                 fontsize=15, fontweight="bold", y=1.01)
    plt.tight_layout()
    _save(fig, "03_km_dfs_radiation.png")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6 · COX PH MODEL
# ══════════════════════════════════════════════════════════════════════════════

def cox_ph_model(df_enc: pd.DataFrame, feat: list):
    _section("SECTION 6 · Cox Proportional Hazards Model")

    cox_df = df_enc[["OS_MONTHS","OS_EVENT_BIN"] + feat].copy()
    for c in cox_df.columns:
        cox_df[c] = pd.to_numeric(cox_df[c], errors="coerce")
    cox_df = cox_df.dropna()

    cph     = CoxPHFitter(penalizer=0.1, l1_ratio=0.1)
    current = [c for c in feat if c in cox_df.columns]

    for attempt in range(7):
        try:
            cph.fit(cox_df[["OS_MONTHS","OS_EVENT_BIN"] + current],
                    duration_col="OS_MONTHS", event_col="OS_EVENT_BIN",
                    show_progress=False)
            print(f"  ✔ Cox fitted on attempt {attempt+1} "
                  f"({len(current)} features, n={len(cox_df)})")
            break
        except Exception as e:
            print(f"  ⚠ Attempt {attempt+1}: {str(e)[:55]}")
            var_s   = cox_df[current].var().sort_values(ascending=False)
            current = var_s.head(max(6, len(current) // 2)).index.tolist()
            if attempt == 6:
                print("  ✖ Cox model failed — skipping."); return None

    summary = cph.summary.copy()
    summary["HR"]       = np.exp(summary["coef"])
    summary["HR_lower"] = np.exp(summary["coef lower 95%"])
    summary["HR_upper"] = np.exp(summary["coef upper 95%"])
    summary["sig"]      = summary["p"].apply(
        lambda p: "***" if p < 0.001 else ("**" if p < 0.01
                  else ("*" if p < 0.05 else "")))

    top_print = (summary.sort_values("p").head(10)
                 [["HR","HR_lower","HR_upper","p","sig"]].round(4))
    print(f"\n  ── Top 10 Cox PH features by p-value ──")
    print(top_print.to_string())

    # ── Forest plot ────────────────────────────────────────────────────────
    top15 = (summary.sort_values("coef", key=abs, ascending=False)
                    .head(15).sort_values("HR"))
    colors_fp = [PAL[3] if hr > 1 else PAL[0] for hr in top15["HR"]]

    fig, ax = plt.subplots(figsize=(11, max(6, len(top15) * 0.58 + 2)))
    y = np.arange(len(top15))
    ax.scatter(top15["HR"], y, color=colors_fp, s=90, zorder=5, edgecolors="white")
    ax.hlines(y, top15["HR_lower"], top15["HR_upper"],
              color=colors_fp, lw=2.2, alpha=0.65)
    ax.axvline(1.0, color="black", ls="--", lw=1.5, alpha=0.7)
    ax.set_yticks(list(y))
    ax.set_yticklabels(top15.index.tolist(), fontsize=8.5)
    ax.set_xlabel("Hazard Ratio (95% CI)", fontsize=11)
    ax.set_title("Cox PH — Hazard Ratios (Top 15 Features)\n"
                 "Red = HR>1 (higher risk)   Blue = HR<1 (protective)",
                 fontweight="bold")
    for i, (_, row) in enumerate(top15.iterrows()):
        if row["sig"]:
            ax.text(row["HR_upper"] + 0.02, i, row["sig"],
                    va="center", fontsize=10, color="black")
    ax.legend(handles=[
        mpatches.Patch(color=PAL[3], label="HR > 1  (increased risk)"),
        mpatches.Patch(color=PAL[0], label="HR < 1  (protective)"),
    ], loc="lower right", fontsize=8)
    plt.tight_layout()
    _save(fig, "04_cox_forest_plot.png")

    # ── Baseline survival + partial effects ────────────────────────────────
    try:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        cph.baseline_survival_.plot(ax=axes[0], color=PAL[2])
        axes[0].set_title("Cox PH — Baseline Survival S₀(t)", fontweight="bold")
        axes[0].set_xlabel("Time (Months)"); axes[0].set_ylim(0, 1.05)

        cph.plot_partial_effects_on_outcome(
            covariates="AGE", values=[30, 45, 60, 75],
            ax=axes[1], cmap=CMAP)
        axes[1].set_title("Partial Effect of Age on OS", fontweight="bold")
        axes[1].set_ylim(0, 1.05)

        plt.suptitle("Cox PH — Baseline & Partial Effects",
                     fontsize=14, fontweight="bold", y=1.01)
        plt.tight_layout()
        _save(fig, "05_cox_baseline_partial.png")
    except Exception as e:
        print(f"  ⚠ Partial effects plot skipped: {e}")

    return cph


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 7 · CLASSIFICATION LABEL
# ══════════════════════════════════════════════════════════════════════════════

def build_label(df: pd.DataFrame) -> pd.DataFrame:
    _section("SECTION 7 · Building 5-Year Survival Classification Label")
    CUTOFF = 60
    df = df.copy()
    df["SURVIVED_5Y"] = np.nan
    df.loc[df["OS_MONTHS"] >= CUTOFF, "SURVIVED_5Y"] = 1
    df.loc[(df["OS_MONTHS"] < CUTOFF) & (df["OS_EVENT_BIN"] == 1),
           "SURVIVED_5Y"] = 0

    n0 = len(df)
    df = df.dropna(subset=["SURVIVED_5Y"])
    df["SURVIVED_5Y"] = df["SURVIVED_5Y"].astype(int)
    vc = df["SURVIVED_5Y"].value_counts().sort_index()

    print(f"  ✔ Retained     : {len(df)} patients")
    print(f"  ✔ Dropped      : {n0 - len(df)} (censored before 60 months — ambiguous)")
    print(f"  ✔ Died < 5 yr  : {vc.get(0,0)}  ({vc.get(0,0)/len(df)*100:.1f}%)")
    print(f"  ✔ Survived ≥5yr: {vc.get(1,0)}  ({vc.get(1,0)/len(df)*100:.1f}%)")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 8 · TRAIN & CROSS-VALIDATE
# ══════════════════════════════════════════════════════════════════════════════

def train_and_evaluate(df_cls: pd.DataFrame, feat: list):
    _section("SECTION 8 · Predictive Modelling (5-Fold CV)")

    f = [c for c in feat if c in df_cls.columns]
    X = df_cls[f].apply(pd.to_numeric, errors="coerce").fillna(0)
    y = df_cls["SURVIVED_5Y"]
    print(f"  ✔ X shape: {X.shape}  |  class ratio: "
          f"{dict(y.value_counts().sort_index())}")

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    models = {
        "Logistic Regression": Pipeline([
            ("sc", StandardScaler()),
            ("clf", LogisticRegression(C=0.1, max_iter=2000,
                                        class_weight="balanced",
                                        random_state=SEED))
        ]),
        "Random Forest": Pipeline([
            ("sc", StandardScaler()),
            ("clf", RandomForestClassifier(n_estimators=400, max_depth=8,
                                            min_samples_leaf=5,
                                            class_weight="balanced",
                                            random_state=SEED, n_jobs=-1))
        ]),
        "Gradient Boosting": Pipeline([
            ("sc", StandardScaler()),
            ("clf", GradientBoostingClassifier(n_estimators=200,
                                                learning_rate=0.05,
                                                max_depth=4, subsample=0.8,
                                                random_state=SEED))
        ]),
        "MLP Neural Net": Pipeline([
            ("sc", StandardScaler()),
            ("clf", MLPClassifier(hidden_layer_sizes=(128, 64, 32),
                                   activation="relu", solver="adam",
                                   alpha=0.01, max_iter=600,
                                   early_stopping=True,
                                   validation_fraction=0.12,
                                   random_state=SEED))
        ]),
    }
    if XGB_OK:
        sw = (y == 0).sum() / (y == 1).sum()
        models["XGBoost"] = Pipeline([
            ("sc", StandardScaler()),
            ("clf", XGBClassifier(n_estimators=200, learning_rate=0.05,
                                   max_depth=4, scale_pos_weight=sw,
                                   use_label_encoder=False,
                                   eval_metric="logloss",
                                   random_state=SEED, n_jobs=-1))
        ])

    scoring = {"accuracy":"accuracy","f1":"f1",
               "roc_auc":"roc_auc","ap":"average_precision"}

    rows, fitted = [], {}
    print(f"\n  {'Model':<25} {'AUC':>7} {'F1':>7} {'Acc':>7} {'PR-AUC':>8}")
    print(f"  {'─'*25} {'─'*7} {'─'*7} {'─'*7} {'─'*8}")

    for name, mdl in models.items():
        cv_r = cross_validate(mdl, X, y, cv=cv, scoring=scoring,
                              return_train_score=False, n_jobs=-1)
        fitted[name] = mdl.fit(X, y)
        auc = cv_r["test_roc_auc"].mean()
        f1  = cv_r["test_f1"].mean()
        acc = cv_r["test_accuracy"].mean()
        ap  = cv_r["test_ap"].mean()
        print(f"  {name:<25} {auc:>7.3f} {f1:>7.3f} {acc:>7.3f} {ap:>8.3f}")
        rows.append({
            "Model"   : name,
            "ROC-AUC" : f"{auc:.3f} ± {cv_r['test_roc_auc'].std():.3f}",
            "F1"      : f"{f1:.3f} ± {cv_r['test_f1'].std():.3f}",
            "Accuracy": f"{acc:.3f} ± {cv_r['test_accuracy'].std():.3f}",
            "PR-AUC"  : f"{ap:.3f} ± {cv_r['test_ap'].std():.3f}",
            "_auc"    : auc,
        })

    comp = (pd.DataFrame(rows)
            .sort_values("_auc", ascending=False)
            .reset_index(drop=True))
    best = comp.iloc[0]["Model"]
    print(f"\n  ✔ Best model by AUC: {best}")
    return comp, fitted, X, y, f


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 9 · EVALUATION PLOTS
# ══════════════════════════════════════════════════════════════════════════════

def plot_evaluation(fitted, X, y, comp):
    _section("SECTION 9 · Evaluation Plots")

    n        = len(fitted)
    col_list = PAL[:n]

    # ── ROC curves ────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, n + 1, figsize=(5 * (n + 1), 5))
    for ax, (name, mdl), col in zip(axes[:-1], fitted.items(), col_list):
        ys = mdl.predict_proba(X)[:, 1]
        fpr, tpr, _ = roc_curve(y, ys)
        auc = roc_auc_score(y, ys)
        ax.plot(fpr, tpr, color=col, lw=2.5)
        ax.fill_between(fpr, tpr, alpha=0.10, color=col)
        ax.plot([0, 1], [0, 1], "k--", lw=1)
        ax.set_title(f"{name}\nAUC={auc:.3f}", fontweight="bold", fontsize=10)
        ax.set_xlabel("FPR"); ax.set_ylabel("TPR")

    ax = axes[-1]
    for (name, mdl), col in zip(fitted.items(), col_list):
        ys = mdl.predict_proba(X)[:, 1]
        fpr, tpr, _ = roc_curve(y, ys)
        auc = roc_auc_score(y, ys)
        ax.plot(fpr, tpr, color=col, lw=2, label=f"{name} ({auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.fill_between([0, 1], [0, 1], alpha=0.04, color="grey")
    ax.set_title("All Models — ROC", fontweight="bold")
    ax.set_xlabel("FPR"); ax.legend(fontsize=7.5)
    plt.suptitle("ROC Curves — 5-Year Survival Prediction",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    _save(fig, "06_roc_curves.png")

    # ── Confusion matrices ─────────────────────────────────────────────────
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4.5))
    if n == 1:
        axes = [axes]
    for ax, (name, mdl) in zip(axes, fitted.items()):
        cm = confusion_matrix(y, mdl.predict(X))
        ConfusionMatrixDisplay(cm, display_labels=["Died<5Y","Surv≥5Y"]).plot(
            ax=ax, colorbar=False, cmap="Blues")
        ax.set_title(name, fontweight="bold", fontsize=10)
    plt.suptitle("Confusion Matrices — 5-Year Survival",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    _save(fig, "07_confusion_matrices.png")

    # ── PR curves ─────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 6))
    baseline = y.mean()
    ax.axhline(baseline, color="grey", ls="--", lw=1.2,
               label=f"Baseline (prev={baseline:.2f})")
    for (name, mdl), col in zip(fitted.items(), col_list):
        ys = mdl.predict_proba(X)[:, 1]
        pr, rc, _ = precision_recall_curve(y, ys)
        ap = average_precision_score(y, ys)
        ax.plot(rc, pr, color=col, lw=2, label=f"{name}  AP={ap:.3f}")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curves — 5-Year Survival", fontweight="bold")
    ax.legend(fontsize=8); ax.set_ylim(0, 1.05)
    plt.tight_layout()
    _save(fig, "08_pr_curves.png")

    # ── Model comparison bar ───────────────────────────────────────────────
    cv2 = comp.copy()
    for m in ["ROC-AUC","F1","Accuracy","PR-AUC"]:
        cv2[m] = cv2[m].apply(lambda x: float(x.split(" ±")[0]))
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(cv2)); w = 0.18
    metrics = ["ROC-AUC","F1","Accuracy","PR-AUC"]
    for j, met in enumerate(metrics):
        bars = ax.bar(x + j * w, cv2[met], w, label=met,
                      color=PAL[j], alpha=0.88, edgecolor="white")
        for b, v in zip(bars, cv2[met]):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.005,
                    f"{v:.3f}", ha="center", fontsize=6.5, rotation=45)
    ax.set_xticks(x + w * 1.5)
    ax.set_xticklabels(cv2["Model"], fontsize=9)
    ax.set_ylabel("Score (5-fold CV mean)"); ax.set_ylim(0, 1.13)
    ax.set_title("Model Comparison — 5-Year Survival Prediction", fontweight="bold")
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    _save(fig, "09_model_comparison.png")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 10 · RISK STRATIFICATION
# ══════════════════════════════════════════════════════════════════════════════

def risk_stratification(df_cls: pd.DataFrame, fitted: dict, feat: list):
    _section("SECTION 10 · Risk Stratification")

    rf = fitted.get("Random Forest") or list(fitted.values())[0]
    f  = [c for c in feat if c in df_cls.columns]
    Xr = df_cls[f].apply(pd.to_numeric, errors="coerce").fillna(0)

    prob = rf.predict_proba(Xr)[:, 1]
    df2  = df_cls.copy()
    df2["RISK_SCORE"] = prob
    df2["RISK_GROUP"] = pd.qcut(prob, q=3,
                                labels=["High Risk","Medium Risk","Low Risk"])

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    group_pal = [PAL[3], PAL[1], PAL[2]]
    Tr, Er = [], []
    for i, grp in enumerate(["High Risk","Medium Risk","Low Risk"]):
        m = df2["RISK_GROUP"] == grp
        if m.sum() < 5:
            continue
        kmf = KaplanMeierFitter()
        kmf.fit(df2.loc[m,"OS_MONTHS"], df2.loc[m,"OS_EVENT_BIN"],
                label=f"{grp} (n={m.sum()})")
        kmf.plot_survival_function(ax=axes[0], ci_show=True,
                                   color=group_pal[i], lw=2.2)
        Tr.append(df2.loc[m,"OS_MONTHS"]); Er.append(df2.loc[m,"OS_EVENT_BIN"])

    if len(Tr) >= 2:
        ct = pd.concat(Tr); ce = pd.concat(Er)
        cg = pd.concat([pd.Series([g] * len(t))
                        for g, t in zip(["High","Mid","Low"][:len(Tr)], Tr)])
        pv_r = multivariate_logrank_test(ct, cg, ce).p_value
        axes[0].set_title(f"Risk-Stratified Survival\nlog-rank p = {pv_r:.4f}",
                          fontweight="bold")
        print(f"  ✔ Risk-stratification log-rank p = {pv_r:.4f}")

    axes[0].set_xlabel("Time (Months)"); axes[0].set_ylabel("Survival Probability")
    axes[0].set_ylim(0, 1.05); axes[0].legend(loc="upper right")

    for i, grp in enumerate(["High Risk","Medium Risk","Low Risk"]):
        m = df2["RISK_GROUP"] == grp
        axes[1].hist(df2.loc[m,"RISK_SCORE"], bins=20,
                     color=group_pal[i], alpha=0.7, label=grp, edgecolor="white")
    axes[1].set_xlabel("Predicted Probability (Survived ≥5Y)")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Risk Score Distribution by Group", fontweight="bold")
    axes[1].legend()

    plt.suptitle("Risk Stratification — RF Predicted Survival Probability",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    _save(fig, "10_risk_stratification.png")
    return df2


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 11 · SHAP EXPLAINABILITY  ← BUG FIXED HERE
# ══════════════════════════════════════════════════════════════════════════════

def shap_explainability(fitted: dict, X: pd.DataFrame, feat: list, y: pd.Series):
    """
    SHAP-based explainability with permutation-importance fallback.

    BUG FIXED: The original permutation_importance call passed
    `X.index.map(lambda i: 0)` as the target — which produced a 2-D array
    when SHAP values were a list, causing:
        ValueError: Data must be 1-dimensional, got ndarray of shape (80, 2)
    Fix: pass the actual label vector `y` to permutation_importance, and
    extract SHAP values for class-1 robustly from both list and ndarray returns.
    """
    _section("SECTION 11 · SHAP Explainability")

    rf_name = "Random Forest" if "Random Forest" in fitted else list(fitted.keys())[0]
    mdl     = fitted[rf_name]
    clf     = mdl.named_steps["clf"]
    scaler  = mdl.named_steps["sc"]

    Xs_arr = scaler.transform(X)
    idx    = np.random.choice(len(Xs_arr), min(400, len(Xs_arr)), replace=False)
    Xs     = pd.DataFrame(Xs_arr[idx], columns=feat)
    y_sub  = y.iloc[idx].reset_index(drop=True)           # ← aligned labels

    if SHAP_OK and hasattr(clf, "estimators_"):
        ex = shap.TreeExplainer(clf)
        sv_raw = ex.shap_values(Xs)

        # ── Robust extraction of class-1 SHAP values ──────────────────────
        # TreeExplainer returns list[ndarray] for multi-output classifiers
        # and ndarray (n_samples, n_features) for binary ones.
        if isinstance(sv_raw, list):
            # Binary RF: sv_raw = [class0_array, class1_array]
            sv = sv_raw[1] if len(sv_raw) > 1 else sv_raw[0]
        else:
            sv = sv_raw          # Already (n_samples, n_features)

        # Validate shape
        if sv.ndim != 2 or sv.shape[1] != len(feat):
            print(f"  ⚠ Unexpected SHAP shape {sv.shape} — falling back to "
                  f"permutation importance")
            sv = None
        else:
            # ── Beeswarm plot ──────────────────────────────────────────────
            fig, ax = plt.subplots(figsize=(11, 8))
            shap.summary_plot(sv, Xs, feature_names=feat,
                              max_display=20, show=False, plot_type="dot")
            plt.title(f"SHAP Feature Importance — {rf_name}\n"
                      "Each dot = 1 patient  |  Colour = feature value",
                      fontweight="bold")
            plt.tight_layout()
            _save(plt.gcf(), "11_shap_beeswarm.png")

            # ── Bar plot of mean |SHAP| ────────────────────────────────────
            mean_sv = np.abs(sv).mean(axis=0)
            top20   = (pd.Series(mean_sv, index=feat)
                       .sort_values(ascending=False).head(20))
            fig, ax = plt.subplots(figsize=(10, 7))
            colors_sh = [PAL[1] if v > top20.median() else PAL[0]
                         for v in top20.values]
            top20.sort_values().plot(kind="barh", ax=ax,
                                     color=list(reversed(colors_sh)),
                                     alpha=0.88, edgecolor="white")
            ax.set_xlabel("Mean |SHAP Value|  (impact on model output)")
            ax.set_title(f"Top 20 Features — {rf_name}\n"
                         "Ranked by mean absolute SHAP value",
                         fontweight="bold")
            plt.tight_layout()
            _save(fig, "12_shap_bar.png")

            # ── Waterfall for highest-risk patient ─────────────────────────
            try:
                # Highest-risk = patient whose survival probability is lowest
                # i.e. largest negative SHAP contribution toward class 1
                row_scores = sv.sum(axis=1)
                hr_idx     = int(np.argmin(row_scores))
                exp_val    = (ex.expected_value[1]
                              if isinstance(ex.expected_value, (list, np.ndarray))
                              else ex.expected_value)
                shap.waterfall_plot(
                    shap.Explanation(
                        values      = sv[hr_idx],
                        base_values = exp_val,
                        data        = Xs.iloc[hr_idx].values,
                        feature_names=feat,
                    ),
                    max_display=15, show=False
                )
                plt.title("SHAP Waterfall — Highest Risk Patient",
                          fontweight="bold")
                plt.tight_layout()
                _save(plt.gcf(), "13_shap_waterfall.png")
            except Exception as e:
                print(f"  ⚠ Waterfall skipped: {e}")

    else:
        sv = None   # triggers permutation importance below

    # ── Permutation importance fallback ───────────────────────────────────
    if sv is None or not SHAP_OK:
        print(f"  ℹ Using permutation importance (SHAP unavailable)")
        # FIX: pass `y` (the true labels), NOT a zero array derived from X.index
        pi = permutation_importance(
            mdl, X, y,               # ← FIXED: y instead of X.index.map(...)
            n_repeats=8,
            random_state=SEED,
            n_jobs=-1,
            scoring="roc_auc"
        )
        # pi.importances_mean is 1-D (one value per feature) — safe to use
        fi = (pd.Series(pi.importances_mean, index=feat)
              .sort_values(ascending=False).head(20))
        fig, ax = plt.subplots(figsize=(10, 7))
        fi.sort_values().plot(kind="barh", ax=ax,
                              color=PAL[2], alpha=0.85, edgecolor="white")
        ax.set_xlabel("Permutation Importance (mean decrease in ROC-AUC)")
        ax.set_title("Top 20 Features — Permutation Importance",
                     fontweight="bold")
        plt.tight_layout()
        _save(fig, "11_permutation_importance.png")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 12 · CALIBRATION CURVES  (NEW — recruiter-impressive addition)
# ══════════════════════════════════════════════════════════════════════════════

def plot_calibration(fitted: dict, X: pd.DataFrame, y: pd.Series):
    _section("SECTION 12 · Calibration Curves")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    n = len(fitted)

    ax = axes[0]
    ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Perfectly calibrated")

    brier_rows = []
    for (name, mdl), col in zip(fitted.items(), PAL[:n]):
        prob = mdl.predict_proba(X)[:, 1]
        fraction_pos, mean_pred = calibration_curve(y, prob, n_bins=10)
        bs = brier_score_loss(y, prob)
        ax.plot(mean_pred, fraction_pos, "s-", color=col, lw=2,
                label=f"{name}  (Brier={bs:.3f})")
        brier_rows.append({"Model": name, "Brier Score": round(bs, 4)})

    ax.set_xlabel("Mean Predicted Probability")
    ax.set_ylabel("Fraction of Positives")
    ax.set_title("Calibration Curves — Reliability Diagram",
                 fontweight="bold")
    ax.legend(fontsize=8)

    # Brier score bar chart
    ax2 = axes[1]
    bs_df = pd.DataFrame(brier_rows).sort_values("Brier Score")
    colors_b = [PAL[i] for i in range(len(bs_df))]
    bars = ax2.barh(bs_df["Model"], bs_df["Brier Score"],
                    color=colors_b, edgecolor="white", alpha=0.88)
    ax2.axvline(0.25, color="grey", ls="--", lw=1.2, label="Uninformative = 0.25")
    for bar, v in zip(bars, bs_df["Brier Score"]):
        ax2.text(v + 0.002, bar.get_y() + bar.get_height() / 2,
                 f"{v:.4f}", va="center", fontsize=9)
    ax2.set_xlabel("Brier Score  (lower = better calibrated)")
    ax2.set_title("Model Calibration — Brier Scores", fontweight="bold")
    ax2.legend(fontsize=8)

    plt.suptitle("Prediction Calibration Analysis — 5-Year Survival",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    _save(fig, "14_calibration_curves.png")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 13 · SUMMARY REPORT
# ══════════════════════════════════════════════════════════════════════════════

def print_summary(df_raw, df_proc, df_cls, comp, cph):
    print("\n" + "═" * 68)
    print("  PIPELINE SUMMARY REPORT")
    print("═" * 68)

    ec   = df_proc["OS_EVENT_BIN"].value_counts().sort_index()
    vc   = df_cls["SURVIVED_5Y"].value_counts().sort_index()
    best = comp.iloc[0]

    report = f"""
  ┌─────────────────────────────────────────────────────────────┐
  │  DATASET                                                    │
  │    Raw patients          : {df_raw.shape[0]:>6,}                       │
  │    After preprocessing   : {df_proc.shape[0]:>6,}                       │
  │    Models evaluated      : {comp.shape[0]:>6}                          │
  │                                                             │
  │  SURVIVAL STATISTICS                                        │
  │    Living (censored)     : {ec.get(0,0):>6,}                       │
  │    Deceased (events)     : {ec.get(1,0):>6,}                       │
  │    Event rate            : {ec.get(1,0)/df_proc.shape[0]*100:>5.1f}%                      │
  │    Median OS             : {df_proc['OS_MONTHS'].median():>5.1f} months                 │
  │                                                             │
  │  5-YEAR CLASSIFICATION                                      │
  │    Died before 5 years   : {vc.get(0,0):>6,}                       │
  │    Survived ≥ 5 years    : {vc.get(1,0):>6,}                       │
  │    Classification n      : {len(df_cls):>6,}                       │
  │                                                             │
  │  BEST MODEL: {best['Model']:<46} │
  │    ROC-AUC               : {best['ROC-AUC']:>20}              │
  │    F1 Score              : {best['F1']:>20}              │
  │    PR-AUC                : {best['PR-AUC']:>20}              │
  │    Accuracy              : {best['Accuracy']:>20}              │
  └─────────────────────────────────────────────────────────────┘
    """
    print(report)

    print("  ALL MODELS (sorted by ROC-AUC):")
    print(f"  {'Model':<25} {'ROC-AUC':>14} {'F1':>14} {'PR-AUC':>14}")
    print(f"  {'─'*25} {'─'*14} {'─'*14} {'─'*14}")
    for _, row in comp.iterrows():
        print(f"  {row['Model']:<25} {row['ROC-AUC']:>14} "
              f"{row['F1']:>14} {row['PR-AUC']:>14}")

    print(f"\n  OUTPUT FILES")
    print(f"  {'─'*40}")
    for p in sorted(PLOTS.glob("*.png")):
        print(f"    plots/{p.name}")
    for p in sorted(DATA.glob("*")):
        print(f"    data/{p.name}")
    print()


# ══════════════════════════════════════════════════════════════════════════════
# MAIN ORCHESTRATOR
# ══════════════════════════════════════════════════════════════════════════════

def run_pipeline(filepath: str):
    t0 = time.time()
    print("═" * 68)
    print("  TCGA-BRCA SURVIVAL ANALYSIS PIPELINE")
    print(f"  File: {filepath}")
    print("═" * 68)

    # 1. Load
    raw = load_and_validate(filepath)

    # 2. Preprocess
    df = preprocess(raw)

    # 3. Feature engineering
    df_enc, feat = engineer_features(df)

    # Save cleaned data
    clean_path = DATA / "tcga_brca_cleaned.csv"
    df_enc.to_csv(clean_path, index=False)
    print(f"\n  ✔ Cleaned dataset saved → {clean_path}")

    # 4. EDA
    plot_eda(df, raw)

    # 5. Kaplan-Meier
    kaplan_meier_analysis(df, raw)

    # 6. Cox PH
    cph = cox_ph_model(df_enc, feat)

    # 7. Classification label
    df_cls = build_label(df_enc)
    df_cls.to_csv(DATA / "tcga_brca_classification.csv", index=False)

    # 8. Train & evaluate
    comp, fitted, X, y, f = train_and_evaluate(df_cls, feat)
    comp.drop("_auc", axis=1).to_csv(DATA / "model_comparison.csv", index=False)
    print(f"  ✔ Model comparison saved → {DATA/'model_comparison.csv'}")

    # 9. Evaluation plots
    plot_evaluation(fitted, X, y, comp)

    # 10. Risk stratification
    df_risk = risk_stratification(df_cls, fitted, f)
    df_risk[["OS_MONTHS","OS_EVENT_BIN","SURVIVED_5Y",
             "RISK_SCORE","RISK_GROUP"]].to_csv(
        DATA / "patient_risk_scores.csv", index=False)

    # 11. SHAP (BUG FIXED — now passes `y` to permutation_importance)
    shap_explainability(fitted, X, f, y)

    # 12. Calibration (NEW)
    plot_calibration(fitted, X, y)

    # 13. Summary
    print_summary(raw, df, df_cls, comp, cph)

    elapsed = time.time() - t0
    print("═" * 68)
    print(f"  PIPELINE COMPLETE  ({elapsed/60:.1f} min)")
    print("═" * 68)
    print(f"\n  All outputs in: {ROOT.resolve()}/\n")


# ══════════════════════════════════════════════════════════════════════════════
# ▶  ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

DATA_FILE = "/content/data_clinical_patient.txt"   # ← Colab default path

if __name__ == "__main__":
    real = [a for a in sys.argv[1:]
            if not a.startswith("-") and not a.endswith(".json") and a]
    if real:
        DATA_FILE = real[0]
    if not os.path.exists(DATA_FILE):
        print(f"\n  ✖ Not found: '{DATA_FILE}'")
        print("  Edit DATA_FILE above and re-run.")
        sys.exit(1)
    run_pipeline(DATA_FILE)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 12.0 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=e489b243b09b4c6878d75dea9fc8b8d8b4805503ccde5cc578b67d5df99821cd
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma
════════════════════════════════════════════════════════════════════
  TCGA-BRCA SURVIVAL ANALYSIS PIPELINE
  File: /content/data_clinical_patient.txt
════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────────────
  SECTION 1 · Loading & Validating Data
────────────────────────────────────────────────────────────────────
  ✔ Loaded       : 1,084 patients × 38 columns
  ✔ File size    : 341,851 bytes
  ✔ All manda

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import shutil

src = "/content/tcga_brca_results"
dst = "/content/drive/MyDrive/tcga_brca_results"

shutil.copytree(src, dst, dirs_exist_ok=True)

print("✅ All results copied to Google Drive!")

✅ All results copied to Google Drive!
